# A realistic e-commerce environment for AI shopping agents
Unified catalogue, navigation taxonomy, SQL and document schemas, 150 graded shopping scenarios.

*Independent case study based on public data · not affiliated with or endorsed by micro1.* Real products are **PUBLIC DATA** (Amazon Berkeley Objects, Open Food Facts); the other-platform export, injected errors, prices, stock and personas are **SIMULATED**. This notebook reads the outputs of the pipeline (see README, *How to rerun*).

In [1]:
import json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT, PROC = ROOT / 'data' / 'outputs', ROOT / 'data' / 'processed'
pd.set_option('display.width', 140)
stats = json.loads((OUT / 'catalog_stats.json').read_text())
cat = pd.read_parquet(PROC / 'catalog.parquet')
{k: stats[k] for k in ('raw_listings', 'by_source', 'clean_skus', 'parents', 'duplicates_merged', 'held_for_review')}

{'raw_listings': 22007,
 'by_source': {'abo': 13165, 'off': 6442, 'mkb': 2400},
 'clean_skus': 19789,
 'parents': 19040,
 'duplicates_merged': 2218,
 'held_for_review': 689}

## 1. Source profile
How each source fills the fields the shop needs (share of SKUs with a value, clean catalogue).

In [2]:
fields = ['brand', 'colour', 'size', 'material', 'length_cm', 'weight_g', 'net_qty', 'description']
cat.groupby('source')[fields].apply(lambda g: g.notna().mean()).round(2)

,brand,colour,size,material,length_cm,weight_g,net_qty,description
source,,,,,,,,
abo,1.00,0.54,0.02,0.29,0.66,0.64,0.09,0.60
mkb,1.00,0.54,0.01,0.28,0.69,0.65,0.07,0.58
off,0.72,0.13,0.00,0.00,0.00,0.00,0.29,0.00


In [3]:
pd.Series(stats['colour_methods']).rename('listings').to_frame().T  # where the clean colour comes from

,missing,field,title,unmapped,placeholder
listings,11225,7824,1377,1173,408


## 2. Quality before and after
26 rules (completeness, validity, consistency, uniqueness), weighted score per SKU.

In [4]:
r = pd.read_csv(OUT / 'quality_rules_before_after.csv')
r.assign(fail_before=(1 - r.pass_rate_before).round(3), fail_after=(1 - r.pass_rate_after).round(3))[['id', 'severity', 'description', 'fail_before', 'fail_after']]

,id,severity,description,fail_before,fail_after
0,C01,error,Title is present,0.000,0.000
1,C02,error,Brand is present,0.082,0.090
2,C03,error,Product is classified in a leaf category,0.229,0.000
3,C04,error,Price is present,0.000,0.000
4,C05,warning,All attributes required by the category are fi...,0.473,0.440
5,C06,info,Description or bullet points are present,0.572,0.586
6,C07,warning,Food products state a net quantity (g or ml),0.670,0.660
7,V01,error,Price is strictly positive,0.001,0.001
8,V02,warning,Colour is in the controlled list,0.983,0.000
9,V03,warning,"Size is on the agreed scale (XXS-3XL, EU shoe ...",NaN,0.000


In [5]:
pd.read_csv(OUT / 'quality_by_category.csv').sort_values('quality').head(8)

,department,category_path,skus,quality,held
65,Grocery & Gourmet Food,Grocery & Gourmet Food > Dairy & Chilled > Yogurt,298,0.8447,3
61,Grocery & Gourmet Food,Grocery & Gourmet Food > Breakfast & Spreads >...,309,0.8590,4
73,Grocery & Gourmet Food,Grocery & Gourmet Food > Snacks & Sweets > Swe...,325,0.8652,25
53,Grocery & Gourmet Food,Grocery & Gourmet Food > Bakery & Fresh > Sand...,302,0.8708,2
57,Grocery & Gourmet Food,Grocery & Gourmet Food > Beverages > Tea,346,0.8708,4
63,Grocery & Gourmet Food,Grocery & Gourmet Food > Dairy & Chilled > Cheese,347,0.8751,10
58,Grocery & Gourmet Food,Grocery & Gourmet Food > Beverages > Water,289,0.8753,21
69,Grocery & Gourmet Food,"Grocery & Gourmet Food > Pantry > Rice, Grains...",349,0.8828,42


## 3. SIMULATED errors: caught and fixed
The answer key is opened only here, after the pipeline ran.

In [6]:
pd.read_csv(OUT / 'detection_by_type.csv')[['label', 'injected', 'caught_rate', 'fixed_rate']]

,label,injected,caught_rate,fixed_rate
0,Colour word outside the list,100,1.000,0.710
1,"Duplicate, reworded title",100,0.950,0.950
2,Same product on 2 platforms,1600,0.863,0.863
3,Misspelled brand,100,0.860,0.670
4,Variant listed as separate product,100,0.810,0.800
5,"Price error (x10, x100, /10, 0)",100,0.740,0.830
6,Product in the wrong category,100,0.650,0.470
7,Required attribute missing,100,0.610,0.320
8,"Wrong unit (cm/in, kg/lb/oz)",100,0.350,0.000


In [7]:
json.loads((OUT / 'detection_summary.json').read_text())

{'injected': 2400,
 'caught': 1978,
 'fixed': 1856,
 'mkb_new_items': 800,
 'wrongly_merged': 40,
 'unit_repair_suggestions': 296,
 'unit_repair_suggestions_on_injected_errors': 18}

## 4. Taxonomy coverage and classification accuracy

In [8]:
pd.read_csv(OUT / 'taxonomy_coverage.csv').sort_values('skus', ascending=False)

,department,skus,leaves_used,leaves_total
4,Grocery & Gourmet Food,7362,27,27
6,Home & Furniture,2552,22,22
2,"Clothing, Shoes & Accessories",2400,26,28
3,Electronics & Accessories,1691,10,10
5,Health & Personal Care,1034,9,9
10,Other,774,1,1
13,Tools & Home Improvement,757,6,6
7,Household Supplies,757,7,7
8,Kitchen & Dining,732,11,11
12,Sports & Outdoors,539,7,7


In [9]:
ev = json.loads((OUT / 'classification_eval.json').read_text())
pd.DataFrame(ev['precision_by_method'])

,category_method,n,precision
0,keyword rule,24,0.833
1,other (low confidence),8,0.000
2,source category,156,0.833
3,text model,5,0.400
4,text model (overrode source),3,0.667
5,text model (within source group),4,0.250


In [10]:
{k: ev[k] for k in ('final_accuracy', 'final_accuracy_unambiguous', 'department_accuracy', 'rules_precision', 'model_precision_proba_0.8', 'other_share_catalogue')}

{'final_accuracy': 0.775,
 'final_accuracy_unambiguous': 0.824,
 'department_accuracy': 0.875,
 'rules_precision': 0.825,
 'model_precision_proba_0.8': 0.908,
 'other_share_catalogue': 0.0391}

In [11]:
pd.read_csv(OUT / 'classification_errors.csv')[['manual_category', 'predicted_category', 'category_method', 'ambiguous_note']].head(10)

,manual_category,predicted_category,category_method,ambiguous_note
0,Other Products,Major Appliances,keyword rule,skateboard: no sports-equipment leaf for boards
1,Sports Balls,Other Products,other (low confidence),NaN
2,"Wall Art, Posters & Decals",Desk Accessories & Whiteboards,source category,NaN
3,Bathroom Accessories,Faucets & Plumbing,source category,NaN
4,Table Linen & Disposables,"Plates, Bowls & Dinnerware Sets",source category,NaN
5,Wallets,Suitcases & Luggage,source category,NaN
6,Door & Cabinet Hardware,Fasteners & Mechanical Parts,source category,NaN
7,Protein & Sports Nutrition,Vitamins & Supplements,source category,NaN
8,Blenders & Other Small Appliances,Other Products,other (low confidence),NaN
9,Blenders & Other Small Appliances,Other Products,other (low confidence),NaN


## 5. Schemas

In [12]:
sc = json.loads((OUT / 'schema_checks.json').read_text())
pd.DataFrame(sc['checks'])

,check,result
0,variants without a product,0
1,products in a non-leaf category,0
2,leaves without a Google id,0
3,SKUs without a price,0
4,SKUs without stock row,0
5,required attributes missing (active SKUs),9225
6,listings merged into another SKU,2218
7,parents with 2+ variants,442
8,max tree depth,4
9,blue sofas under $700 in stock (filter query a...,11


In [13]:
{k: sc[k] for k in ('documents', 'documents_invalid', 'avg_variants_per_document', 'row_counts')}

{'documents': 19040,
 'documents_invalid': 1,
 'avg_variants_per_document': 1.039,
 'row_counts': {'brand': 2290,
  'brand_alias': 2758,
  'category': 212,
  'attribute': 17,
  'category_attribute': 581,
  'product': 19040,
  'variant': 19789,
  'price': 19789,
  'inventory': 19789,
  'variant_attribute': 19907,
  'source_listing': 22007}}

## 6. Scenarios and calibration on real baskets (UCI Online Retail II)

In [14]:
cal = json.loads((OUT / 'basket_calibration.json').read_text())
{k: cal[k] for k in ('invoices', 'period', 'share_small_baskets', 'median_lines_small_baskets', 'share_lines_qty1', 'cancelled_line_share')}

{'invoices': 40077,
 'period': '2009-12 to 2011-12',
 'share_small_baskets': 0.1152,
 'median_lines_small_baskets': 1.0,
 'share_lines_qty1': 0.2817,
 'cancelled_line_share': 0.0183}

In [15]:
pd.read_csv(OUT / 'scenario_summary.csv').groupby('type').agg(n=('id', 'size'), difficulty=('difficulty', 'first'), median_acceptable=('n_acceptable', 'median'), with_trap=('has_trap', 'mean')).round(2)

,n,difficulty,median_acceptable,with_trap
type,,,,
ambiguous,20,4,1.0,1.0
basket,20,4,496.5,1.0
comparison,20,3,1.0,1.0
multi_constraint,30,2,3.0,0.9
no_answer,15,5,0.0,1.0
simple_search,25,1,1.0,0.2
substitution,20,3,123.5,1.0


## 7. Reference agents in the environment

In [16]:
res = pd.read_csv(OUT / 'agent_results.csv')
res.groupby(['agent', 'view']).agg(success=('success', 'mean'), score=('score', 'mean'), steps=('steps', 'mean')).round(3)

success  score  steps
agent           view                                 
keyword         1 raw             0.207  0.415  3.000
                2 + attributes    0.220  0.429  3.000
                3 + taxonomy      0.207  0.413  3.000
                4 clean           0.213  0.418  3.000
keyword+filters 1 raw             0.293  0.442  3.413
                2 + attributes    0.420  0.561  3.573
                3 + taxonomy      0.513  0.645  3.660
                4 clean           0.560  0.673  3.660

In [17]:
res[res.view == '4 clean'].pivot_table(index='type', columns='agent', values='success').round(2)

agent,keyword,keyword+filters
type,,
ambiguous,0.00,0.60
basket,0.00,0.25
comparison,0.00,0.25
multi_constraint,0.10,0.60
no_answer,0.00,0.60
simple_search,0.88,0.84
substitution,0.35,0.70


In [18]:
f = res[(res.view == '4 clean') & ~res.success].failure.str.split('; ').explode().str.replace(r':.*', '', regex=True)
f.value_counts().head(10)

failure
constraint broken                            107
did not ask the clarifying question           27
bought something although nothing matches     21
reported no match although one exists         13
role 'soft drink' not satisfied               10
role 'phone case' not satisfied                9
role 'chocolate' not satisfied                 8
role 'sandwich' not satisfied                  8
role 'bookcase' not satisfied                  7
role 'coffee' not satisfied                    6
Name: count, dtype: int64

**Reading**: the same simple agent succeeds far more often on the clean catalogue than on the raw listings: normalised attributes make filters work, the taxonomy makes category queries work, and merging duplicates and holding price errors removes wrong answers. What is left fails on reasoning (ambiguous requests, baskets), which is what a training environment should isolate.